In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
cd /content/drive/MyDrive/project/

/content/drive/.shortcut-targets-by-id/1IhtrClOvYkKUSZxwUoLOhxwHY9CdXCaH/project


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.2 MB/s eta 0:00:00


In [ ]:
cat ./tcdata/dataset.yaml


names:
- '0'
- '1'
- '2'
- '3'
- '4'
- '5'
- '6'
- '7'
- '8'
- '9'
nc: 10
path: ./tcdata
train: images/mchar_train
val: images/mchar_val


In [ ]:
import torch
torch.cuda.is_available()


True

In [8]:
from ultralytics import YOLO

model = YOLO("runs/detect/digits_yolo8/weights/last.pt")  # pesos preentrenados

model.train(

    data='./tcdata/dataset.yaml',
    epochs=30,
    imgsz=512,
    batch=64,

    cos_lr=True,    # Cosinne Annealing

    device='0',
    patience=20,  # Early stopping

    # Loss tuning
    # total loss = box_loss + clas_loss + dlf_loss
    box=7.5,
    dfl=2.5,   # Distribution Focal Loss (Objetos pequeños)
    cls=0.6,
    kobj=1.3,     # Controlo falsos positivos

    freeze = 0,

    # Training behaviour
    lr0=0.001,
    pretrained=True,
    optimizer='AdamW',
    weight_decay=0.0001,  # Weight decay para AdamW´


    # Data augmentaion

    mosaic= 1,
    scale= 0.7,
    degrees= 0.0,


    name='digits_yolo8',
    project='runs/detect',
    exist_ok=True,
)


Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.6, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./tcdata/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=2.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.3, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/digits_yolo8/weights/last.pt, momentum=0.937, mosaic=1, multi_scale=False, name=digits_yolo8, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=20, perspective=0.

KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/digits_yolo8/weights/best.pt")


# Fine Tuning

"""
model.train(
    data='./tcdata/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device='0',
    patience=15,

    # Loss tuning
    box=7.5,
    dfl=2.0,
    cls=0.6,
    kobj=1.3,

    # Training behaviour
    lr0=0.001,
    freeze=10,
    pretrained=True,
    augment=True,

    name='digits_yolo8',
    project='runs/detect',
    exist_ok=True,
)
"""

#Second try generating results

In [ ]:
# Define Project root
from google.colab import drive
drive.mount('/content/drive/')


In [ ]:
cd /content/drive/MyDrive/project/

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import sys
import os

# --- PATH SETUP ---
PROJECT_ROOT = os.getcwd()

In [ ]:
def predict_number(r):


  """ For each prediction, crop the bb and predict the number, change its class"""

  if r.boxes is None or len(r.boxes) == 0:
      return None

  boxes = r.boxes.xyxy.cpu().numpy()
  classes = r.boxes.cls.cpu().numpy().astype(int)
  confs = r.boxes.conf.cpu().numpy()

  detections = []
  for box, cls, conf in zip(boxes, classes, confs):
      x1, _, x2, _ = box

      detections.append({
          "digit": cls,
          "x_center": (x1 + x2) / 2,
          "conf": conf
      })

  # ordenar de izquierda a derecha
  detections.sort(key=lambda x: x["x_center"])

  number_str = "".join(str(d["digit"]) for d in detections)

  return int(number_str)

In [ ]:
import os
import zipfile
import cv2
import numpy as np
from ultralytics import YOLO


def predict():

    PROJECT_ROOT = os.getcwd()

    zip_file = os.path.join(PROJECT_ROOT, "tcdata", "mchar_test_a.zip")
    assert os.path.exists(zip_file), "ZIP not found"

    model = YOLO("runs/detect/digits_yolo8/weights/best.pt")

    results_csv = []

    images = []
    names = []

    with zipfile.ZipFile(zip_file) as z:
      for name in sorted(z.namelist()):

          if name.endswith("/"):
              continue
          if not name.lower().endswith((".png", ".jpg", ".jpeg")):
              continue

          data = z.read(name)
          img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)

          if img is None:
              continue

          img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

          images.append(img)
          names.append(os.path.basename(name))

    BATCH = 32
    for i in range(0, len(images), BATCH):
        batch_imgs = images[i:i+BATCH]
        batch_names = names[i:i+BATCH]

        # increasing imgsz to increase resolution and reduce confidence
        preds = model(batch_imgs, conf=0.1)

        for fname, r in zip(batch_names, preds):
            number = predict_number(r)
            results_csv.append(f"{fname},{number if number else ''}")

        print(f"Processed {i+BATCH}/{len(images)}")

    out_file = os.path.join(PROJECT_ROOT, "prediction_result", "resultOLGA2.csv")
    with open(out_file, "w") as f:
      f.write("file_name,file_code\n")
      f.write("\n".join(results_csv))

    print(f"\nDone. Results saved to {out_file}")




In [ ]:
model = YOLO("runs/detect/digits_yolo8/weights/best.pt")

In [ ]:
test_set = os.path.join(PROJECT_ROOT, "tcdata", "images", "mchar_test_a")
image_path = os.path.join(test_set, "000000.png")

In [ ]:
import cv2
from PIL import Image
import matplotlib.pyplot as plt

# Realizar predicción
results = model(test_set+"/000010.png")

# Mostrar resultados automáticamente
results[0].show()  # Esto abrirá la imagen con los bounding boxes

In [ ]:
predict()